In [0]:
%pip install torch

# IMPORTS

In [0]:
import torch
print(torch.__version__)
print(f"GPU disponible: {torch.cuda.is_available()}")

In [0]:
import requests
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import collect_list,struct
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data load

In [0]:
prices = spark.read.format('delta').load('/Volumes/market-mood/features/feature_store')

In [0]:
prices.printSchema()

# Format data for entry

In [0]:
context_size = 20
w = Window.partitionBy('ticker').orderBy('date').rowsBetween(-(context_size-1),0)
prices = prices.withColumn('context',collect_list(struct('close','volume','log_return','rsi','macd','bb_position','volatility','vol_ratio')).over(w))

In [0]:
fs_pdf = prices.dropna().toPandas()#convertir a pandas para poder usarlo para entrenar
fs_pdf = fs_pdf.sort_values(["ticker", "date"]).reset_index(drop=True)

In [0]:
print(f"Shape: {fs_pdf.shape}")
print(f"Tickers: {fs_pdf['ticker'].nunique()}")
print(f"Rango: {fs_pdf['date'].min()} → {fs_pdf['date'].max()}")

In [0]:
fs_pdf.info()
fs_pdf['date'] = pd.to_datetime(fs_pdf['date']) 

In [0]:
FEATURE_COLS = ["close", "volume", "log_return", "rsi", "macd", 
                "bb_position", "volatility", "vol_ratio"]
TARGET_COL = "target"
SEQUENCE_LENGTH = 20

# Split temporal
train = fs_pdf[fs_pdf["date"] < "2023-01-01"]
val   = fs_pdf[(fs_pdf["date"] >= "2023-01-01") & (fs_pdf["date"] < "2024-01-01")]
test  = fs_pdf[fs_pdf["date"] >= "2024-01-01"]

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

In [0]:
def change_format(context_list):
    return np.array([[v for v in d.values()] for d in context_list])

# Convertir context a arrays (20, 8)
train["context"] = train["context"].map(change_format)
val["context"]   = val["context"].map(change_format)
test["context"]  = test["context"].map(change_format)

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

In [0]:
scaler = StandardScaler()
